# ChemProp evaluation across standard, reaction-grouped, and OOD splits

This notebook tunes target-specific ChemProp settings on one fixed random 8:1:1 split, then holds those settings fixed for the 8:1:1 test set, random five-fold evaluation, reaction-pair-grouped five-fold evaluation, barrier-ordered folds, and OOD evaluation. Every evaluation uses the same target normalization, internal 90:10 fit/validation split, model construction, and metrics.


In [1]:
from pathlib import Path
import os
import warnings

import numpy as np
import pandas as pd
import torch
from lightning import pytorch as pl
from chemprop import data, featurizers, models, nn
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.model_selection import KFold

warnings.filterwarnings("ignore")


In [2]:
def find_repository_root(start=None):
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in (start, *start.parents):
        required = candidate / "Data" / "DFT_Result" / "Active_Energy_for_AttFP.csv"
        if required.is_file() and (candidate / "Code").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the CycloAddGenerator repository from the current directory.")

REPO_ROOT = find_repository_root()
os.chdir(REPO_ROOT)
DATA_FILE = REPO_ROOT / "Data" / "DFT_Result" / "Active_Energy_for_AttFP.csv"
TARGETS = ["deltaGa(Solvent)", "Diene_Distort", "Ene_Distort", "Interaction"]
SMILES_COLUMN = "smiles"
BATCH_SIZE = 256
MAX_EPOCHS = 20
TUNING_SEED = 0
CV_SEED = 0
OOD_SEEDS = [0, 3, 6]

if not DATA_FILE.is_file():
    raise FileNotFoundError(DATA_FILE)
df_input = pd.read_csv(DATA_FILE).reset_index(drop=True)
required_columns = [SMILES_COLUMN, "Diene_Index", "Ene_Index", *TARGETS]
missing = [column for column in required_columns if column not in df_input.columns]
if missing:
    raise KeyError(f"Missing required input columns: {missing}")
smiles = df_input[SMILES_COLUMN].astype(str).to_numpy()
all_datapoints = {
    target: [data.ReactionDatapoint.from_smi(smi.replace(".", ">"), [y])
             for smi, y in zip(smiles, df_input[target].to_numpy())]
    for target in TARGETS
}
print(f"Loaded {len(df_input):,} reactions from {DATA_FILE.relative_to(REPO_ROOT)}")


Loaded 8,273 reactions from Data\DFT_Result\Active_Energy_for_AttFP.csv


## Shared split and ChemProp evaluation helpers

Hyperparameters are selected by validation RMSE on the fixed 8:1:1 tuning split. Test data do not participate in selection. For every reported split, the training portion is shuffled once and divided into 90% model-fitting data and 10% validation data; target scaling is fitted on the fitting data and reused for validation and test data.


In [3]:
AGGREGATIONS = {
    "MeanAggregation": nn.MeanAggregation,
    "SumAggregation": nn.SumAggregation,
    "NormAggregation": nn.NormAggregation,
}
REACTION_MODES = list(featurizers.RxnMode.keys())


def make_811_split(n_rows, seed=0):
    permutation = np.random.RandomState(seed).permutation(n_rows)
    n_train = int(0.8 * n_rows)
    n_valid = int(0.1 * n_rows)
    return (permutation[:n_train], permutation[n_train:n_train + n_valid],
            permutation[n_train + n_valid:])


def split_fit_validation(train_ids, seed=0):
    shuffled = np.asarray(train_ids, dtype=int).copy()
    np.random.RandomState(seed).shuffle(shuffled)
    n_fit = int(0.9 * len(shuffled))
    return shuffled[:n_fit], shuffled[n_fit:]


def score_predictions(y_true, y_pred):
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)
    return {
        "r2": float(r2_score(y_true, y_pred)),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
    }


def fit_predict(target, config, fit_ids, valid_ids, test_ids=None, seed=0,
                batch_size=BATCH_SIZE, max_epochs=MAX_EPOCHS):
    pl.seed_everything(int(seed), workers=True)
    datapoints = all_datapoints[target]
    featurizer = featurizers.CondensedGraphOfReactionFeaturizer(mode_=config["mode"])
    train_data = [datapoints[int(i)] for i in fit_ids]
    valid_data = [datapoints[int(i)] for i in valid_ids]
    train_dset = data.ReactionDataset(train_data, featurizer)
    scaler = train_dset.normalize_targets()
    valid_dset = data.ReactionDataset(valid_data, featurizer)
    valid_dset.normalize_targets(scaler)

    train_loader = data.build_dataloader(train_dset, batch_size=batch_size, num_workers=0)
    valid_loader = data.build_dataloader(valid_dset, batch_size=batch_size, num_workers=0, shuffle=False)

    mp = nn.BondMessagePassing(*featurizer.shape)
    agg = AGGREGATIONS[config["agg"]]()
    output_transform = nn.UnscaleTransform.from_standard_scaler(scaler)
    ffn = nn.RegressionFFN(output_transform=output_transform)
    mpnn = models.MPNN(
        mp, agg, ffn, True,
        [nn.metrics.RMSE(), nn.metrics.MAE(), nn.metrics.R2Score()],
    )
    trainer = pl.Trainer(
        logger=False, enable_checkpointing=False, enable_progress_bar=False,
        accelerator="auto", devices=1, max_epochs=max_epochs,
    )
    trainer.fit(mpnn, train_loader, valid_loader)
    with torch.inference_mode():
        valid_pred = np.concatenate(trainer.predict(mpnn, valid_loader), axis=0).reshape(-1)
        test_pred = None
        test_scores = None
        if test_ids is not None:
            test_data = [datapoints[int(i)] for i in test_ids]
            test_dset = data.ReactionDataset(test_data, featurizer)
            test_loader = data.build_dataloader(test_dset, batch_size=batch_size, num_workers=0, shuffle=False)
            test_pred = np.concatenate(trainer.predict(mpnn, test_loader), axis=0).reshape(-1)
            test_true = df_input[target].to_numpy()[test_ids]
            test_scores = score_predictions(test_true, test_pred)
    valid_true = df_input[target].to_numpy()[valid_ids]
    valid_scores = score_predictions(valid_true, valid_pred)
    return valid_scores, test_scores, test_pred


def evaluate_one_split(target, config, train_ids, test_ids, seed, split_name, fold_id=None):
    fit_ids, valid_ids = split_fit_validation(train_ids, seed=seed + (fold_id or 0))
    valid_scores, test_scores, _ = fit_predict(
        target, config, fit_ids, valid_ids, test_ids, seed=seed + (fold_id or 0)
    )
    return {
        "split": split_name, "fold_id": fold_id, "seed": seed,
        "target": target, "aggregation": config["agg"],
        "reaction_featurizer": str(config["mode"]),
        "n_fit": len(fit_ids), "n_valid": len(valid_ids),
        "n_test": len(test_ids),
        "valid_rmse": valid_scores["rmse"],
        **test_scores,
    }


## 1. Tune each target on the fixed random 8:1:1 split

The same split is used for all target/configuration comparisons. Candidate settings are ranked by validation RMSE; the held-out test set is evaluated only after the best setting has been selected.


In [4]:
tune_train_ids, tune_valid_ids, tune_test_ids = make_811_split(len(df_input), TUNING_SEED)
tuning_rows = []
best_target_configs = {}

for target in TARGETS:
    candidates = []
    for mode in REACTION_MODES:
        for agg_name in AGGREGATIONS:
            config = {"mode": mode, "agg": agg_name}
            valid_scores, _, _ = fit_predict(
                target, config, tune_train_ids, tune_valid_ids, test_ids=None,
                seed=TUNING_SEED,
            )
            row = {
                "target": target, "reaction_featurizer": str(mode),
                "aggregation": agg_name, **{f"valid_{k}": v for k, v in valid_scores.items()},
            }
            tuning_rows.append(row)
            candidates.append((row["valid_rmse"], row["valid_mae"], config))
    _, _, best_config = min(candidates, key=lambda item: (item[0], item[1]))
    best_target_configs[target] = best_config

tuning_results = pd.DataFrame(tuning_rows).sort_values(["target", "valid_rmse", "valid_mae"])
best_config_table = pd.DataFrame([
    {"target": target, "reaction_featurizer": str(config["mode"]),
     "aggregation": config["agg"]}
    for target, config in best_target_configs.items()
])
display(best_config_table)
display(tuning_results.groupby("target", as_index=False).head(5))


Seed set to 0
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loading `train_dataloader` to estimate number of stepping batches.

  | Name            | Type               | Params | Mode 
---------------------------------------------------------------
0 | message_passing | BondMessagePassing | 252 K  | train
1 | agg             | MeanAggregation    | 0      | train
2 | bn              | BatchNorm1d        | 600    | train
3 | predictor       | RegressionFFN      | 90.6 K | train
4 | X_d_transform   | Identity           | 0      | train
5 | metrics         | ModuleList         | 0      | train
---------------------------------------------------------------
343 K     Trainable params
0         Non-trainable params
343 K     Total params
1.374     Total estimated model params size (MB)
26        Modules in train mode
0         Modules in eval mode
`Trainer.fit` stopped: `max_epoch

,target,reaction_featurizer,aggregation
0,deltaGa(Solvent),REAC_DIFF_BALANCE,MeanAggregation
1,Diene_Distort,REAC_DIFF,MeanAggregation
2,Ene_Distort,REAC_DIFF_BALANCE,SumAggregation
3,Interaction,PROD_DIFF_BALANCE,MeanAggregation


,target,reaction_featurizer,aggregation,valid_r2,valid_mae,valid_rmse
24,Diene_Distort,REAC_DIFF,MeanAggregation,0.927216,1.226275,1.632226
25,Diene_Distort,REAC_DIFF,SumAggregation,0.926627,1.227936,1.638822
28,Diene_Distort,REAC_DIFF_BALANCE,SumAggregation,0.925649,1.260559,1.649710
27,Diene_Distort,REAC_DIFF_BALANCE,MeanAggregation,0.924967,1.243765,1.657251
30,Diene_Distort,PROD_DIFF,MeanAggregation,0.924198,1.249128,1.665728
46,Ene_Distort,REAC_DIFF_BALANCE,SumAggregation,0.766387,1.318401,1.835710
52,Ene_Distort,PROD_DIFF_BALANCE,SumAggregation,0.757903,1.348077,1.868749
43,Ene_Distort,REAC_DIFF,SumAggregation,0.754781,1.340076,1.880761
51,Ene_Distort,PROD_DIFF_BALANCE,MeanAggregation,0.753622,1.333855,1.885198
45,Ene_Distort,REAC_DIFF_BALANCE,MeanAggregation,0.753239,1.346404,1.886661


## 2. Initial random 8:1:1 holdout

The selected target-specific settings are held fixed. This cell reports the untouched 10% test set from the tuning split.


In [5]:
holdout_rows = []
for target, config in best_target_configs.items():
    fit_ids, valid_ids = tune_train_ids, tune_valid_ids
    valid_scores, test_scores, _ = fit_predict(
        target, config, fit_ids, valid_ids, tune_test_ids, seed=TUNING_SEED
    )
    holdout_rows.append({
        "split": "Random 8:1:1", "seed": TUNING_SEED, "target": target,
        "aggregation": config["agg"], "reaction_featurizer": str(config["mode"]),
        "n_fit": len(fit_ids), "n_valid": len(valid_ids), "n_test": len(tune_test_ids),
        "valid_rmse": valid_scores["rmse"], **test_scores,
    })
holdout_results = pd.DataFrame(holdout_rows)
display(holdout_results)


Seed set to 0
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loading `train_dataloader` to estimate number of stepping batches.

  | Name            | Type               | Params | Mode 
---------------------------------------------------------------
0 | message_passing | BondMessagePassing | 252 K  | train
1 | agg             | MeanAggregation    | 0      | train
2 | bn              | BatchNorm1d        | 600    | train
3 | predictor       | RegressionFFN      | 90.6 K | train
4 | X_d_transform   | Identity           | 0      | train
5 | metrics         | ModuleList         | 0      | train
---------------------------------------------------------------
343 K     Trainable params
0         Non-trainable params
343 K     Total params
1.374     Total estimated model params size (MB)
26        Modules in train mode
0         Modules in eval mode
`Trainer.fit` stopped: `max_epoch

,split,seed,target,aggregation,reaction_featurizer,n_fit,n_valid,n_test,valid_rmse,r2,mae,rmse
0,Random 8:1:1,0,deltaGa(Solvent),MeanAggregation,REAC_DIFF_BALANCE,6618,827,828,2.646303,0.886875,1.956647,2.663381
1,Random 8:1:1,0,Diene_Distort,MeanAggregation,REAC_DIFF,6618,827,828,1.642727,0.915054,1.255929,1.764394
2,Random 8:1:1,0,Ene_Distort,SumAggregation,REAC_DIFF_BALANCE,6618,827,828,1.869795,0.739757,1.422006,1.959660
3,Random 8:1:1,0,Interaction,MeanAggregation,PROD_DIFF_BALANCE,6618,827,828,2.365537,0.761890,1.687314,2.388481


## 3. Random five-fold cross-validation

All folds use the same selected target-specific settings and shared fit/validation/evaluation helper.


In [6]:
random_folds = [
    (train_ids, test_ids)
    for train_ids, test_ids in KFold(n_splits=5, shuffle=True, random_state=CV_SEED).split(np.arange(len(df_input)))
]
random_fold_rows = []
for target, config in best_target_configs.items():
    for fold_id, (train_ids, test_ids) in enumerate(random_folds):
        random_fold_rows.append(evaluate_one_split(
            target, config, train_ids, test_ids, CV_SEED, "Random 5-fold", fold_id
        ))
random_fold_results = pd.DataFrame(random_fold_rows)
display(random_fold_results)
display(random_fold_results.groupby("target")[["r2", "mae", "rmse"]].agg(["mean", "std"]))


Seed set to 0
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loading `train_dataloader` to estimate number of stepping batches.

  | Name            | Type               | Params | Mode 
---------------------------------------------------------------
0 | message_passing | BondMessagePassing | 252 K  | train
1 | agg             | MeanAggregation    | 0      | train
2 | bn              | BatchNorm1d        | 600    | train
3 | predictor       | RegressionFFN      | 90.6 K | train
4 | X_d_transform   | Identity           | 0      | train
5 | metrics         | ModuleList         | 0      | train
---------------------------------------------------------------
343 K     Trainable params
0         Non-trainable params
343 K     Total params
1.374     Total estimated model params size (MB)
26        Modules in train mode
0         Modules in eval mode
`Trainer.fit` stopped: `max_epoch

,split,fold_id,seed,target,aggregation,reaction_featurizer,n_fit,n_valid,n_test,valid_rmse,r2,mae,rmse
0,Random 5-fold,0,0,deltaGa(Solvent),MeanAggregation,REAC_DIFF_BALANCE,5956,662,1655,2.569044,0.887076,1.952873,2.628615
1,Random 5-fold,1,0,deltaGa(Solvent),MeanAggregation,REAC_DIFF_BALANCE,5956,662,1655,2.751666,0.891690,1.907053,2.569069
2,Random 5-fold,2,0,deltaGa(Solvent),MeanAggregation,REAC_DIFF_BALANCE,5956,662,1655,2.650739,0.881311,1.969426,2.720136
3,Random 5-fold,3,0,deltaGa(Solvent),MeanAggregation,REAC_DIFF_BALANCE,5957,662,1654,2.561943,0.891978,1.888614,2.556698
4,Random 5-fold,4,0,deltaGa(Solvent),MeanAggregation,REAC_DIFF_BALANCE,5957,662,1654,2.525474,0.878901,2.000432,2.729006
5,Random 5-fold,0,0,Diene_Distort,MeanAggregation,REAC_DIFF,5956,662,1655,1.811580,0.908313,1.385867,1.841896
6,Random 5-fold,1,0,Diene_Distort,MeanAggregation,REAC_DIFF,5956,662,1655,1.810508,0.906615,1.359229,1.876459
7,Random 5-fold,2,0,Diene_Distort,MeanAggregation,REAC_DIFF,5956,662,1655,1.827085,0.906506,1.396061,1.853793
8,Random 5-fold,3,0,Diene_Distort,MeanAggregation,REAC_DIFF,5957,662,1654,1.708589,0.915837,1.315486,1.795174
9,Random 5-fold,4,0,Diene_Distort,MeanAggregation,REAC_DIFF,5957,662,1654,1.741503,0.919790,1.245364,1.714204


r2                 mae                rmse          
                      mean       std      mean       std      mean       std
target                                                                      
Diene_Distort     0.911412  0.006051  1.340401  0.061588  1.816305  0.064323
Ene_Distort       0.752766  0.010393  1.390310  0.048918  1.890919  0.071154
Interaction       0.751384  0.012994  1.734926  0.030019  2.403004  0.056595
deltaGa(Solvent)  0.886191  0.005947  1.943680  0.045667  2.640705  0.081304

## 4. Reaction-pair-grouped five-fold cross-validation

Reactions sharing the same `(Diene_Index, Ene_Index)` pair are assigned to the same fold, independent of row order. Pair groups are deterministically shuffled and greedily assigned to the currently smallest fold to keep fold sizes balanced.


In [7]:
def reaction_pair_grouped_folds(frame, n_splits=5, seed=0):
    group_map = {}
    for row_id, (diene_id, ene_id) in enumerate(zip(frame["Diene_Index"], frame["Ene_Index"])):
        group_map.setdefault((int(diene_id), int(ene_id)), []).append(row_id)
    groups = [np.asarray(ids, dtype=int) for ids in group_map.values()]
    rng = np.random.RandomState(seed)
    order = rng.permutation(len(groups))
    order = sorted(order, key=lambda idx: -len(groups[idx]))
    fold_rows = [[] for _ in range(n_splits)]
    fold_sizes = np.zeros(n_splits, dtype=int)
    for group_idx in order:
        fold_id = int(np.argmin(fold_sizes))
        fold_rows[fold_id].extend(groups[group_idx].tolist())
        fold_sizes[fold_id] += len(groups[group_idx])
    all_ids = np.arange(len(frame))
    folds = []
    for rows in fold_rows:
        test_ids = np.asarray(sorted(rows), dtype=int)
        train_ids = np.setdiff1d(all_ids, test_ids, assume_unique=True)
        folds.append((train_ids, test_ids))
    return folds

reaction_folds = reaction_pair_grouped_folds(df_input, n_splits=5, seed=CV_SEED)
# Verify that no reactant pair is split between train and test in any fold.
for train_ids, test_ids in reaction_folds:
    train_pairs = set(zip(df_input.iloc[train_ids]["Diene_Index"], df_input.iloc[train_ids]["Ene_Index"]))
    test_pairs = set(zip(df_input.iloc[test_ids]["Diene_Index"], df_input.iloc[test_ids]["Ene_Index"]))
    assert train_pairs.isdisjoint(test_pairs)

reaction_fold_rows = []
for target, config in best_target_configs.items():
    for fold_id, (train_ids, test_ids) in enumerate(reaction_folds):
        reaction_fold_rows.append(evaluate_one_split(
            target, config, train_ids, test_ids, CV_SEED, "Reaction-pair-grouped 5-fold", fold_id
        ))
reaction_fold_results = pd.DataFrame(reaction_fold_rows)
display(reaction_fold_results)
display(reaction_fold_results.groupby("target")[["r2", "mae", "rmse"]].agg(["mean", "std"]))


Seed set to 0
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loading `train_dataloader` to estimate number of stepping batches.

  | Name            | Type               | Params | Mode 
---------------------------------------------------------------
0 | message_passing | BondMessagePassing | 252 K  | train
1 | agg             | MeanAggregation    | 0      | train
2 | bn              | BatchNorm1d        | 600    | train
3 | predictor       | RegressionFFN      | 90.6 K | train
4 | X_d_transform   | Identity           | 0      | train
5 | metrics         | ModuleList         | 0      | train
---------------------------------------------------------------
343 K     Trainable params
0         Non-trainable params
343 K     Total params
1.374     Total estimated model params size (MB)
26        Modules in train mode
0         Modules in eval mode
`Trainer.fit` stopped: `max_epoch

,split,fold_id,seed,target,aggregation,reaction_featurizer,n_fit,n_valid,n_test,valid_rmse,r2,mae,rmse
0,Reaction-pair-grouped 5-fold,0,0,deltaGa(Solvent),MeanAggregation,REAC_DIFF_BALANCE,5956,662,1655,2.551286,0.803361,2.521670,3.332014
1,Reaction-pair-grouped 5-fold,1,0,deltaGa(Solvent),MeanAggregation,REAC_DIFF_BALANCE,5956,662,1655,2.585312,0.816515,2.606978,3.585055
2,Reaction-pair-grouped 5-fold,2,0,deltaGa(Solvent),MeanAggregation,REAC_DIFF_BALANCE,5956,662,1655,2.461080,0.748690,2.796197,3.941900
3,Reaction-pair-grouped 5-fold,3,0,deltaGa(Solvent),MeanAggregation,REAC_DIFF_BALANCE,5957,662,1654,2.599571,0.746832,2.747816,3.810636
4,Reaction-pair-grouped 5-fold,4,0,deltaGa(Solvent),MeanAggregation,REAC_DIFF_BALANCE,5957,662,1654,2.740443,0.794328,2.582775,3.483225
5,Reaction-pair-grouped 5-fold,0,0,Diene_Distort,MeanAggregation,REAC_DIFF,5956,662,1655,1.726719,0.873708,1.636608,2.116461
6,Reaction-pair-grouped 5-fold,1,0,Diene_Distort,MeanAggregation,REAC_DIFF,5956,662,1655,1.607867,0.868968,1.748057,2.305751
7,Reaction-pair-grouped 5-fold,2,0,Diene_Distort,MeanAggregation,REAC_DIFF,5956,662,1655,1.710555,0.839726,1.745554,2.419172
8,Reaction-pair-grouped 5-fold,3,0,Diene_Distort,MeanAggregation,REAC_DIFF,5957,662,1654,1.593172,0.778465,1.974929,2.848816
9,Reaction-pair-grouped 5-fold,4,0,Diene_Distort,MeanAggregation,REAC_DIFF,5957,662,1654,1.798515,0.852549,1.728216,2.305615


r2                 mae                rmse          
                      mean       std      mean       std      mean       std
target                                                                      
Diene_Distort     0.842683  0.038352  1.766673  0.125059  2.399163  0.273855
Ene_Distort       0.636647  0.036088  1.704179  0.066719  2.286399  0.105225
Interaction       0.670737  0.014918  2.057347  0.054475  2.760379  0.059166
deltaGa(Solvent)  0.781945  0.032194  2.651087  0.115947  3.630566  0.246104

## 5. Barrier-ordered five-fold evaluation

The barrier-ordered split is retained as a separate analysis. Each target uses the same tuned configuration as the other evaluations.


In [8]:
def barrier_ordered_folds(barrier_values, n_splits=5):
    sorted_ids = np.argsort(np.asarray(barrier_values))
    fold_sizes = np.full(n_splits, len(sorted_ids) // n_splits, dtype=int)
    fold_sizes[:len(sorted_ids) % n_splits] += 1
    all_ids = np.arange(len(sorted_ids))
    folds, start = [], 0
    for fold_size in fold_sizes:
        test_ids = np.sort(sorted_ids[start:start + fold_size])
        train_ids = np.setdiff1d(all_ids, test_ids, assume_unique=True)
        folds.append((train_ids, test_ids))
        start += fold_size
    return folds

barrier_values = df_input["deltaGa(Solvent)"].to_numpy()
barrier_folds = barrier_ordered_folds(barrier_values, n_splits=5)
barrier_fold_rows = []
barrier_fold_predictions = {}
for target, config in best_target_configs.items():
    target_predictions = np.full(len(df_input), np.nan, dtype=float)
    for fold_id, (train_ids, test_ids) in enumerate(barrier_folds):
        fit_ids, valid_ids = split_fit_validation(train_ids, seed=CV_SEED + fold_id)
        _, test_scores, predictions = fit_predict(
            target, config, fit_ids, valid_ids, test_ids, seed=CV_SEED + fold_id
        )
        target_predictions[test_ids] = predictions
        barrier_fold_rows.append({
            "split": "Barrier-ordered 5-fold", "fold_id": fold_id, "target": target,
            "n_test": len(test_ids), "barrier_min": float(np.min(barrier_values[test_ids])),
            "barrier_max": float(np.max(barrier_values[test_ids])), **test_scores,
        })
    barrier_fold_predictions[target] = target_predictions
barrier_fold_results = pd.DataFrame(barrier_fold_rows)
low_barrier_ids = barrier_folds[0][1]
low_barrier_results = pd.DataFrame([
    {"target": target, "n_test": len(low_barrier_ids),
     "barrier_min": float(np.min(barrier_values[low_barrier_ids])),
     "barrier_max": float(np.max(barrier_values[low_barrier_ids])),
     **score_predictions(df_input[target].to_numpy()[low_barrier_ids], preds[low_barrier_ids])}
    for target, preds in barrier_fold_predictions.items()
])
display(barrier_fold_results)
display(barrier_fold_results.groupby("target")[["r2", "mae", "rmse"]].mean())
display(low_barrier_results)


Seed set to 0
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loading `train_dataloader` to estimate number of stepping batches.

  | Name            | Type               | Params | Mode 
---------------------------------------------------------------
0 | message_passing | BondMessagePassing | 252 K  | train
1 | agg             | MeanAggregation    | 0      | train
2 | bn              | BatchNorm1d        | 600    | train
3 | predictor       | RegressionFFN      | 90.6 K | train
4 | X_d_transform   | Identity           | 0      | train
5 | metrics         | ModuleList         | 0      | train
---------------------------------------------------------------
343 K     Trainable params
0         Non-trainable params
343 K     Total params
1.374     Total estimated model params size (MB)
26        Modules in train mode
0         Modules in eval mode
`Trainer.fit` stopped: `max_epoch

,split,fold_id,target,n_test,barrier_min,barrier_max,r2,mae,rmse
0,Barrier-ordered 5-fold,0,deltaGa(Solvent),1655,2.308818,32.525671,-2.182143,6.569126,8.081851
1,Barrier-ordered 5-fold,1,deltaGa(Solvent),1655,32.527948,35.941285,-7.437176,2.184267,2.818416
2,Barrier-ordered 5-fold,2,deltaGa(Solvent),1655,35.941405,39.702527,-6.068293,2.293676,2.891318
3,Barrier-ordered 5-fold,3,deltaGa(Solvent),1654,39.704082,45.336040,-3.817492,2.838382,3.566943
4,Barrier-ordered 5-fold,4,deltaGa(Solvent),1654,45.354884,65.940577,-4.022595,6.629569,8.040281
5,Barrier-ordered 5-fold,0,Diene_Distort,1655,2.308818,32.525671,0.320873,2.596278,3.541247
6,Barrier-ordered 5-fold,1,Diene_Distort,1655,32.527948,35.941285,0.743467,1.295309,1.741243
7,Barrier-ordered 5-fold,2,Diene_Distort,1655,35.941405,39.702527,0.725194,1.493547,2.080093
8,Barrier-ordered 5-fold,3,Diene_Distort,1654,39.704082,45.336040,0.771047,1.674716,2.196875
9,Barrier-ordered 5-fold,4,Diene_Distort,1654,45.354884,65.940577,0.158947,3.207391,4.493710


,r2,mae,rmse
target,,,
Diene_Distort,0.543905,2.053448,2.810633
Ene_Distort,0.419435,1.916051,2.544585
Interaction,0.591660,2.146624,3.003142
deltaGa(Solvent),-4.705540,4.103004,5.079762


,target,n_test,barrier_min,barrier_max,r2,mae,rmse
0,deltaGa(Solvent),1655,2.308818,32.525671,-2.182143,6.569126,8.081851
1,Diene_Distort,1655,2.308818,32.525671,0.320873,2.596278,3.541247
2,Ene_Distort,1655,2.308818,32.525671,0.189074,2.428138,3.256000
3,Interaction,1655,2.308818,32.525671,0.554219,2.607865,3.630290


## 6. Single-cold and double-cold OOD evaluation

OOD split membership follows the same 10% reactant holdout rule and seeds (0, 3, and 6) used for the main-model analysis. In double-cold evaluation, mixed held-out/seen reactant pairs are excluded from both training and testing. The tuned target-specific settings remain fixed; OOD test scores are not used for model selection.


In [9]:
def ood_train_test(frame, seed=0, holdout="both"):
    rng = np.random.RandomState(seed)
    diene_ids = np.sort(frame["Diene_Index"].unique())
    ene_ids = np.sort(frame["Ene_Index"].unique())
    rng.shuffle(diene_ids)
    rng.shuffle(ene_ids)
    held_dienes = set(diene_ids[:len(diene_ids) // 10])
    held_enes = set(ene_ids[:len(ene_ids) // 10])
    diene_held = frame["Diene_Index"].isin(held_dienes).to_numpy()
    ene_held = frame["Ene_Index"].isin(held_enes).to_numpy()
    if holdout == "diene":
        test_mask = diene_held
        train_mask = ~diene_held
    elif holdout == "ene":
        test_mask = ene_held
        train_mask = ~ene_held
    elif holdout == "both":
        test_mask = diene_held & ene_held
        train_mask = ~diene_held & ~ene_held
    else:
        raise ValueError(f"Unknown OOD split: {holdout}")
    return np.flatnonzero(train_mask), np.flatnonzero(test_mask)

OOD_SPLITS = [
    ("Diene single-cold", "diene"),
    ("Ene single-cold", "ene"),
    ("Double-cold", "both"),
]
ood_rows = []
for split_label, holdout in OOD_SPLITS:
    for seed in OOD_SEEDS:
        train_ids, test_ids = ood_train_test(df_input, seed=seed, holdout=holdout)
        for target, config in best_target_configs.items():
            row = evaluate_one_split(target, config, train_ids, test_ids, seed, split_label)
            row["n_excluded"] = len(df_input) - len(train_ids) - len(test_ids)
            ood_rows.append(row)
ood_results = pd.DataFrame(ood_rows)
ood_summary = (
    ood_results.groupby(["split", "target"], as_index=False)
    .agg(seeds=("seed", "nunique"), n_fit=("n_fit", "mean"),
         n_valid=("n_valid", "mean"), n_test=("n_test", "mean"),
         n_excluded=("n_excluded", "mean"),
         r2_mean=("r2", "mean"), r2_std=("r2", "std"),
         mae_mean=("mae", "mean"), mae_std=("mae", "std"),
         rmse_mean=("rmse", "mean"), rmse_std=("rmse", "std"))
)
display(ood_results)
display(ood_summary)


Seed set to 0
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loading `train_dataloader` to estimate number of stepping batches.

  | Name            | Type               | Params | Mode 
---------------------------------------------------------------
0 | message_passing | BondMessagePassing | 252 K  | train
1 | agg             | MeanAggregation    | 0      | train
2 | bn              | BatchNorm1d        | 600    | train
3 | predictor       | RegressionFFN      | 90.6 K | train
4 | X_d_transform   | Identity           | 0      | train
5 | metrics         | ModuleList         | 0      | train
---------------------------------------------------------------
343 K     Trainable params
0         Non-trainable params
343 K     Total params
1.374     Total estimated model params size (MB)
26        Modules in train mode
0         Modules in eval mode
`Trainer.fit` stopped: `max_epoch

,split,fold_id,seed,target,aggregation,reaction_featurizer,n_fit,n_valid,n_test,valid_rmse,r2,mae,rmse,n_excluded
0,Diene single-cold,None,0,deltaGa(Solvent),MeanAggregation,REAC_DIFF_BALANCE,6489,722,1062,2.364867,0.705339,3.878930,5.361784,0
1,Diene single-cold,None,0,Diene_Distort,MeanAggregation,REAC_DIFF,6489,722,1062,1.743230,0.687024,3.408889,4.491611,0
2,Diene single-cold,None,0,Ene_Distort,SumAggregation,REAC_DIFF_BALANCE,6489,722,1062,1.920293,0.608074,1.819365,2.375946,0
3,Diene single-cold,None,0,Interaction,MeanAggregation,PROD_DIFF_BALANCE,6489,722,1062,2.456606,0.629127,2.051765,2.795548,0
4,Diene single-cold,None,3,deltaGa(Solvent),MeanAggregation,REAC_DIFF_BALANCE,7000,778,495,2.626696,0.534928,4.899581,6.072689,0
5,Diene single-cold,None,3,Diene_Distort,MeanAggregation,REAC_DIFF,7000,778,495,1.779165,0.576784,3.794630,4.679361,0
6,Diene single-cold,None,3,Ene_Distort,SumAggregation,REAC_DIFF_BALANCE,7000,778,495,1.958752,0.614769,1.733724,2.261644,0
7,Diene single-cold,None,3,Interaction,MeanAggregation,PROD_DIFF_BALANCE,7000,778,495,2.505024,0.655716,2.086949,2.788157,0
8,Diene single-cold,None,6,deltaGa(Solvent),MeanAggregation,REAC_DIFF_BALANCE,6495,722,1056,2.530945,0.607149,3.676142,4.583020,0
9,Diene single-cold,None,6,Diene_Distort,MeanAggregation,REAC_DIFF,6495,722,1056,1.772501,0.385623,3.914584,4.817352,0


,split,target,seeds,n_fit,n_valid,n_test,n_excluded,r2_mean,r2_std,mae_mean,mae_std,rmse_mean,rmse_std
0,Diene single-cold,Diene_Distort,3,6661.333333,740.666667,871.000000,0.0,0.549811,0.152500,3.706034,0.264233,4.662775,0.163503
1,Diene single-cold,Ene_Distort,3,6661.333333,740.666667,871.000000,0.0,0.626229,0.025866,1.756095,0.055572,2.291783,0.073859
2,Diene single-cold,Interaction,3,6661.333333,740.666667,871.000000,0.0,0.617765,0.044728,2.047026,0.042492,2.777204,0.025640
3,Diene single-cold,deltaGa(Solvent),3,6661.333333,740.666667,871.000000,0.0,0.615805,0.085535,4.151551,0.655700,5.339164,0.745092
4,Double-cold,Diene_Distort,3,6053.333333,673.333333,91.333333,1455.0,0.558802,0.193736,3.690942,0.652952,4.614426,0.408520
5,Double-cold,Ene_Distort,3,6053.333333,673.333333,91.333333,1455.0,0.617901,0.046327,1.619633,0.112597,2.067680,0.180724
6,Double-cold,Interaction,3,6053.333333,673.333333,91.333333,1455.0,0.517659,0.041234,2.057902,0.647196,2.723485,0.844469
7,Double-cold,deltaGa(Solvent),3,6053.333333,673.333333,91.333333,1455.0,0.529465,0.173588,4.428455,0.717752,5.811747,0.343104
8,Ene single-cold,Diene_Distort,3,6755.000000,751.333333,766.666667,0.0,0.802646,0.054925,1.994070,0.227975,2.819699,0.486026
9,Ene single-cold,Ene_Distort,3,6755.000000,751.333333,766.666667,0.0,0.477761,0.108586,1.969941,0.101250,2.674635,0.250462


## 7. Exportable result tables

The detailed per-fold/per-seed results and the corresponding summaries are kept in separate DataFrames so they can be exported without rerunning or manually copying notebook output.


In [10]:
evaluation_summary = pd.concat([
    random_fold_results.assign(split="Random 5-fold"),
    reaction_fold_results.assign(split="Reaction-pair-grouped 5-fold"),
], ignore_index=True)
cv_summary = (
    evaluation_summary.groupby(["split", "target"], as_index=False)
    .agg(folds=("fold_id", "nunique"),
         r2_mean=("r2", "mean"), r2_std=("r2", "std"),
         mae_mean=("mae", "mean"), mae_std=("mae", "std"),
         rmse_mean=("rmse", "mean"), rmse_std=("rmse", "std"))
)
display(cv_summary)
# Optional exports:
# tuning_results.to_csv(REPO_ROOT / "Data/DFT_Result/chemprop_tuning_results.csv", index=False)
# holdout_results.to_csv(REPO_ROOT / "Data/DFT_Result/chemprop_811_results.csv", index=False)
# random_fold_results.to_csv(REPO_ROOT / "Data/DFT_Result/chemprop_random_5fold_results.csv", index=False)
# reaction_fold_results.to_csv(REPO_ROOT / "Data/DFT_Result/chemprop_reaction_grouped_5fold_results.csv", index=False)
# barrier_fold_results.to_csv(REPO_ROOT / "Data/DFT_Result/chemprop_barrier_ordered_results.csv", index=False)
# ood_results.to_csv(REPO_ROOT / "Data/DFT_Result/chemprop_ood_results.csv", index=False)


,split,target,folds,r2_mean,r2_std,mae_mean,mae_std,rmse_mean,rmse_std
0,Random 5-fold,Diene_Distort,5,0.911412,0.006051,1.340401,0.061588,1.816305,0.064323
1,Random 5-fold,Ene_Distort,5,0.752766,0.010393,1.390310,0.048918,1.890919,0.071154
2,Random 5-fold,Interaction,5,0.751384,0.012994,1.734926,0.030019,2.403004,0.056595
3,Random 5-fold,deltaGa(Solvent),5,0.886191,0.005947,1.943680,0.045667,2.640705,0.081304
4,Reaction-pair-grouped 5-fold,Diene_Distort,5,0.842683,0.038352,1.766673,0.125059,2.399163,0.273855
5,Reaction-pair-grouped 5-fold,Ene_Distort,5,0.636647,0.036088,1.704179,0.066719,2.286399,0.105225
6,Reaction-pair-grouped 5-fold,Interaction,5,0.670737,0.014918,2.057347,0.054475,2.760379,0.059166
7,Reaction-pair-grouped 5-fold,deltaGa(Solvent),5,0.781945,0.032194,2.651087,0.115947,3.630566,0.246104
